# LogHawk 🦅: AI-Powered Security Log Analyzer

**IS2083 Advanced Scripting · Lab 5, Part 1**

LogHawk reads security log entries and asks the class **Qwen Coder** model to label each one
**BENIGN**, **SUSPICIOUS**, or **MALICIOUS** with a one-sentence reason, then writes a Markdown
security assessment report.

> **Security:** the API key is loaded from a local `.env` file that is git-ignored. It never appears in this notebook.
> Raw security logs also stay local; only the generated `assessment.md` report is published.

## Part A · Step 3: Install the openai library

In [ ]:
%pip install -q openai

## Part B · Step 4: Connect to the Qwen Coder model

`BASE_URL` is the class GPUStack server (it ends in `/v1`). Both it and the API key are read from a local,
git-ignored `.env` file, so neither is typed into (or committed with) this notebook.

In [ ]:
import os
from pathlib import Path
from openai import OpenAI

def load_env(path=".env"):
    """Read KEY=VALUE lines from a local .env file into environment variables."""
    p = Path(path)
    if p.exists():
        for line in p.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip().strip('"'))

load_env()
BASE_URL = os.environ.get("GPUSTACK_BASE_URL", "REPLACE-WITH-GPUSTACK-URL")  # class model server (from .env)
API_KEY = os.environ.get("GPUSTACK_API_KEY", "REPLACE-WITH-YOUR-KEY")         # never hard-code it
if "REPLACE" in BASE_URL or "REPLACE" in API_KEY:
    raise RuntimeError("Missing .env file: add GPUSTACK_BASE_URL and GPUSTACK_API_KEY (see README).")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Ask the server which models it serves, and pick the Qwen *Coder* model by name
# (safer than a fixed index like data[1], because the list order can change).
models = [m.id for m in client.models.list().data]
print("Models on server:", models)
MODEL = next(m for m in models if "coder" in m.lower())
print("Connected. Model:", MODEL)

## Step 5: Reusable model helper (with retry)

Wraps one chat completion. Retries with increasing sleeps so one flaky network call doesn't crash a long run.
Uses `temperature=1.0` and `top_p=0.95` as the lab specifies.

In [ ]:
import time

def ask_model(prompt: str, temperature: float = 1.0, retries: int = 2) -> str:
    """Send one prompt to Qwen Coder and return its text reply, retrying on failure."""
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                top_p=0.95,
            )
            return resp.choices[0].message.content
        except Exception:
            if attempt == retries:
                raise
            time.sleep(1.5 * (attempt + 1))

print(ask_model("Say hello in one short sentence."))

## Part C · Steps 6–7: Plan the tool, using Qwen Coder as a design assistant

**Plan:** one log line goes in; a label (BENIGN / SUSPICIOUS / MALICIOUS) and a one-sentence reason come out.
Below, Qwen Coder is asked to sketch the design. Its answer is a *first draft* to read critically, not code to paste blindly.

In [ ]:
question = (
    "I am writing a Python function triage_log(line) that asks an LLM to classify a "
    "security log line and returns a dataclass with label and reason. Suggest a robust "
    "prompt format and a defensive way to parse the reply. Keep it concise."
)
print(ask_model(question))

## Part D · Step 8: Model the output, then write the tool

* `Label(str, Enum)` pins the only allowed labels (plus `UNKNOWN` as a safe fallback).
* `Triage` is a typed record instead of a loose dict.
* Two compiled regexes find the `LABEL:` and `REASON:` lines anywhere in the reply (case-insensitive, multi-line).

In [ ]:
from dataclasses import dataclass
from enum import Enum
import re

class Label(str, Enum):
    BENIGN = "BENIGN"
    SUSPICIOUS = "SUSPICIOUS"
    MALICIOUS = "MALICIOUS"
    UNKNOWN = "UNKNOWN"

@dataclass
class Triage:
    line: str
    label: Label
    reason: str

LABEL_RE = re.compile(r"^LABEL:\s*(\w+)", re.IGNORECASE | re.MULTILINE)
REASON_RE = re.compile(r"^REASON:\s*(.+)", re.IGNORECASE | re.MULTILINE)

**The tool itself:** a strict prompt, one `ask_model` call, then defensive parsing that falls back to
`Label.UNKNOWN` (and the raw reply as the reason) when the model ignores the format.

In [ ]:
def triage_log(line: str) -> Triage:
    """LogHawk core: classify one security log line with Qwen Coder."""
    prompt = (
        "You are a security log triage assistant.\n"
        "Classify the log line as BENIGN, SUSPICIOUS, or MALICIOUS and give a one "
        "sentence reason.\n"
        "Reply in EXACTLY this format:\n"
        "LABEL: <BENIGN|SUSPICIOUS|MALICIOUS>\n"
        "REASON: <one sentence>\n\n"
        f"Log line:\n{line}\n"
    )
    text = ask_model(prompt)

    label_match = LABEL_RE.search(text)
    raw = label_match.group(1).upper() if label_match else "UNKNOWN"
    label = Label(raw) if raw in Label.__members__ else Label.UNKNOWN

    reason_match = REASON_RE.search(text)
    reason = reason_match.group(1).strip() if reason_match else text.strip()

    return Triage(line=line, label=label, reason=reason)

## Part E · Step 9: Name the tool

In [ ]:
TOOL_NAME = "LogHawk"   # my tool's portfolio name (also the GitHub repo name)

## Step 10: Quick sanity check on one known-bad line

In [ ]:
print(triage_log("Failed password for root from 203.0.113.9 port 4022 ssh2"))

## Step 11: Load the exported security log

Exported on macOS with:

```bash
log show --predicate 'process == "sudo" OR process == "sshd" OR process == "loginwindow" OR eventMessage CONTAINS[c] "authentication"' --last 6h --style syslog > security_log.txt
```

The raw log stays on this machine (it is git-ignored). Only the report is published, and `redact()`
hides this Mac's hostname and username in anything printed or written to it.

In [ ]:
import getpass, socket

LOG_FILE = "security_log.txt"     # Windows CSV would be "security_log.csv"
MAX_ENTRIES = 20                  # keep this first version small and quick

# Privacy: hide this Mac's hostname and username in anything printed or published
_PRIVATE = {
    socket.gethostname().split(".")[0]: "<HOST>",
    getpass.getuser(): "<USER>",
}

def redact(text: str) -> str:
    for secret, placeholder in _PRIVATE.items():
        if secret:
            text = text.replace(secret, placeholder)
    return text

with open(LOG_FILE, encoding="utf-8", errors="ignore") as f:
    raw_lines = f.readlines()
print(f"Found: {LOG_FILE} ({os.path.getsize(LOG_FILE):,} bytes, {len(raw_lines):,} lines)")
for ln in raw_lines[1:4]:                      # peek at a few entries (redacted)
    print("  ", redact(ln.rstrip())[:140])

## Step 12: Assess the file and write the report

`read_log_lines` keeps only real log entries (lines that start with a timestamp; this skips headers and
multi-line debug dumps), then takes up to `MAX_ENTRIES` **spread evenly across the whole file**, so the
sample covers the full 6-hour window instead of just the first few seconds.

In [ ]:
ENTRY_RE = re.compile(r"^\d{4}-\d{2}-\d{2}[ T]")   # a real entry starts with a date

def read_log_lines(path, limit=MAX_ENTRIES):
    with open(path, encoding="utf-8", errors="ignore") as f:
        lines = [ln.strip() for ln in f if ENTRY_RE.match(ln)]
    if len(lines) <= limit:
        return lines
    step = len(lines) / limit                    # evenly spaced sample across the file
    return [lines[int(i * step)] for i in range(limit)]

entries = read_log_lines(LOG_FILE)
results = [triage_log(line) for line in entries]
print(f"Assessed {len(results)} entries from {LOG_FILE}")

**Report writer:** a summary table of label counts, then one finding per entry. Every log line and reason
passes through `redact()` (from Step 11) so the public report doesn't leak machine or account names.

In [ ]:
from collections import Counter
from datetime import datetime

def write_report(results, tool_name, path="assessment.md"):
    counts = Counter(r.label.value for r in results)
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"# {tool_name} Security Log Assessment\n\n")
        f.write(f"Generated: {datetime.now():%Y-%m-%d %H:%M}  \n")
        f.write(f"Model: {MODEL}  \n")
        f.write(f"Entries assessed: {len(results)}\n\n")
        f.write("## Summary\n\n| Label | Count |\n|---|---|\n")
        for lab in Label:
            f.write(f"| {lab.value} | {counts.get(lab.value, 0)} |\n")
        f.write("\n## Findings\n\n")
        for r in results:
            f.write(f"- **{r.label.value}**: {redact(r.reason)}\n")
            f.write(f"  `{redact(r.line)[:220]}`\n")
    return path

report_path = write_report(results, TOOL_NAME)
print("Wrote", report_path, "\n")
print(open(report_path, encoding="utf-8").read())

## Step 13: Reflect: do I understand and trust it?

*(Answer in your own words; these are the questions from the handout.)*

**1. Why model the output with an Enum and dataclass instead of a bare dict?**

YOUR ANSWER HERE

**2. What does the defensive parsing do when the model ignores the format?**

YOUR ANSWER HERE

**3. Which entries in the report look wrong, and how would you catch that?**

YOUR ANSWER HERE

---
*AI assistance used: Qwen Coder (class model) for tool design; Claude for project setup and code review.*